**Azure Virtual Machines** is Azure's IaaS compute service — it lets you run Windows or Linux servers in the cloud without managing physical hardware. VMs are the foundation of many Azure architectures, and understanding them deeply is essential before moving to higher-level services like App Service, AKS, or Azure Functions.

Azure VMs give you full control over the OS, installed software, and configuration — you are responsible for patching, scaling, and availability.

## What You Configure at Deployment

| Setting | Description |
|---|---|
| **Image** | The OS image (Windows Server, Ubuntu, RHEL, custom) |
| **Size** | CPU + memory + temp disk combination (e.g. `Standard_D2s_v5`) |
| **Authentication** | SSH key pair (Linux) or username/password; Azure AD login available |
| **Virtual Network** | VNet and subnet the VM's NIC attaches to |
| **Public IP** | Optional static or dynamic public IP address |
| **NSG** | Network Security Group — firewall rules at NIC or subnet level |
| **OS disk** | Managed disk for the root volume — type and size |
| **Data disks** | Additional managed disks attached after the OS disk |
| **Managed Identity** | Assigns a system or user identity so the VM can access other Azure services |
| **Custom data** | Script or cloud-init config that runs on first boot |

## VM Images

An **image** is a template containing the OS, optional pre-installed software, and disk layout.

### Image sources

| Source | Description |
|---|---|
| **Azure Marketplace** | First-party (Microsoft) and third-party images — Windows Server, Ubuntu, RHEL, SQL Server, etc. |
| **Azure Compute Gallery** | Your own custom images, versioned and replicated across regions |
| **Managed Image** | A one-off captured image from an existing VM (simpler but less flexible than Compute Gallery) |

### Azure Compute Gallery

The recommended way to manage custom images at scale:
- Store **versioned image definitions** (e.g. `my-app-image/1.0.0`, `my-app-image/1.1.0`)
- Automatically **replicate** image versions to multiple regions
- Share images **across subscriptions and tenants** via RBAC
- Support for both generalised (sysprepped) and specialised images

> **Generalised**: OS is cleaned of machine-specific data (hostname, users) — used to create many identical VMs.  
> **Specialised**: Exact copy of a VM including its state — used for disaster recovery or migration.

## VM Sizes

VM sizes follow the naming pattern: `Standard_` + `Series` + `vCPUs` + `features` + `_version`

Examples: `Standard_D4s_v5`, `Standard_E8ds_v5`, `Standard_F2s_v2`

**Feature suffixes in the name:**
- `s` — supports Premium SSD storage
- `d` — includes a local temporary NVMe disk
- `a` — AMD processor
- `l` — lower memory ratio
- `m` — memory-intensive variant

### VM series families

| Series | Optimised for | AWS equivalent | Use cases |
|---|---|---|---|
| **B** | Burstable — baseline CPU with credits | t3/t4g | Dev/test, low-traffic apps, CI runners |
| **D** | General purpose — balanced CPU/memory | m6i/m7i | Web servers, app tiers, small databases |
| **E** | Memory optimised | r6i/r7i | In-memory databases, SAP HANA, large caches |
| **F** | Compute optimised — high CPU ratio | c6i/c7i | Batch processing, gaming, ML inference |
| **L** | Storage optimised — high IOPS/throughput | i4i | NoSQL, data warehouses, Elasticsearch |
| **M** | Very large memory | x2iedn | SAP HANA scale-up, in-memory analytics |
| **N (NC/ND/NV)** | GPU | p4/g5 | ML training (NC), ML inference (ND), visualisation (NV) |
| **H** | High-performance compute | hpc7g | CFD, molecular dynamics, MPI workloads |

### B-series burstable VMs

B-series VMs run at a **baseline CPU percentage** and earn credits when below baseline. Credits are spent when bursting above. Ideal for workloads with occasional spikes — the same concept as AWS T-series instances.

## VM Disks

Every Azure VM has at least two disk concepts:

| Disk | Persists? | Description |
|---|---|---|
| **OS disk** | Yes | Root volume — contains the OS; always a managed disk |
| **Temporary disk** | No | Local SSD physically attached to the host — lost on stop/redeploy/resize |
| **Data disks** | Yes | Additional managed disks you attach — for application data, databases |

> The temporary disk (mounted as `/dev/sdb` on Linux, `D:` on Windows) is **not** for anything you want to keep. Use it only for swap, caches, or scratch space.

### Managed Disk types

| Type | Class | Max IOPS | Best for |
|---|---|---|---|
| **Standard HDD** | HDD | 500 IOPS/disk | Backups, dev/test, infrequent access |
| **Standard SSD** | SSD | 6,000 IOPS/disk | Web servers, lightly-used apps |
| **Premium SSD** | SSD | 20,000 IOPS/disk | Production databases, I/O-intensive apps |
| **Premium SSD v2** | SSD | 80,000 IOPS/disk | High-performance DBs with independent IOPS/throughput tuning |
| **Ultra Disk** | SSD | 160,000 IOPS/disk | SAP HANA, top-tier SQL, sub-millisecond latency requirements |

### Disk encryption

- **Server-Side Encryption (SSE)**: Enabled by default — data encrypted at rest using platform-managed or customer-managed keys
- **Azure Disk Encryption (ADE)**: Encrypts the OS and data disks using BitLocker (Windows) or DM-Crypt (Linux) — keys stored in Azure Key Vault

## VM Networking

Every Azure VM has at least one **Network Interface Card (NIC)**. Each NIC:
- Is attached to a **subnet** within a VNet
- Has a **private IP** (static or dynamic from the subnet range)
- Optionally has a **public IP** associated
- Has one or more **NSGs** attached (at the NIC or subnet level)

### Public IP types

| Type | Behaviour |
|---|---|
| **Dynamic** | Assigned from a pool — changes if the VM is stopped/deallocated |
| **Static** | Fixed IP — does not change; billed even when unassociated |

### Network Security Groups (NSGs)

An NSG is a stateful packet filter — it allows or denies traffic based on rules. Each rule has:
- **Priority** (100–4096) — lower number = evaluated first
- **Direction** — Inbound or Outbound
- **Protocol** — TCP, UDP, ICMP, or Any
- **Source / Destination** — IP, CIDR, service tag, or application security group
- **Action** — Allow or Deny

NSGs can be attached at two levels:
- **Subnet level** — applies to all VMs in the subnet
- **NIC level** — applies to one specific VM

When both exist, traffic must pass **both** NSGs.

### Default NSG rules

Every NSG has built-in rules that cannot be deleted (priority 65000+):
- Allow inbound from same VNet
- Allow inbound from Azure Load Balancer
- Deny all other inbound traffic
- Allow all outbound to internet
- Allow outbound within VNet
- Deny all other outbound

### Application Security Groups (ASGs)

Instead of managing IP ranges in NSG rules, group VMs into **Application Security Groups** (e.g. `asg-web`, `asg-app`, `asg-db`) and write rules referencing the group names. VMs can belong to multiple ASGs.

## Custom Data & Extensions

### Custom Data / cloud-init

Pass a script or cloud-init YAML at VM creation time. It runs **once on first boot**.

```bash
#!/bin/bash
apt-get update -y
apt-get install -y nginx
systemctl start nginx
systemctl enable nginx
echo "<h1>Hello from Azure VM</h1>" > /var/www/html/index.html
```

### VM Extensions

Extensions are small applications that run post-deployment to configure or automate tasks on a running VM.

| Extension | Purpose |
|---|---|
| **Custom Script Extension** | Run an arbitrary script on a running VM |
| **Azure Monitor Agent** | Send logs and metrics to Log Analytics / Azure Monitor |
| **Microsoft Entra ID login** | Allow SSH/RDP using Entra ID credentials instead of local accounts |
| **Desired State Configuration (DSC)** | Enforce a PowerShell DSC configuration on Windows VMs |
| **Dependency Agent** | Enable service map and VM insights |
| **Anti-Malware** | Microsoft Antimalware for Windows VMs |

## Pricing & Purchasing Options

| Option | Commitment | Savings | Best for |
|---|---|---|---|
| **Pay-as-you-go** | None | 0% | Short-term, unpredictable workloads |
| **Reserved VM Instances (1yr)** | 1 year | ~40% | Steady workloads running 24/7 |
| **Reserved VM Instances (3yr)** | 3 years | ~60–72% | Long-running stable workloads |
| **Azure Savings Plan** | 1 or 3 years | up to 65% | Flexible — applies to compute across regions/sizes |
| **Spot VMs** | None | up to 90% | Fault-tolerant, interruptible workloads |
| **Azure Hybrid Benefit** | Existing licence | up to 49% | Customers with on-premises Windows Server or SQL Server licences |
| **Dev/Test pricing** | Active subscription | Significant | Non-production VMs for Visual Studio subscribers |

### Spot VMs

Azure Spot VMs use spare capacity at a steep discount. Azure can **evict** them with a 30-second warning when capacity is needed (unlike AWS's 2-minute notice). Set an **eviction policy**:
- **Deallocate** — VM is stopped but disks are preserved (can restart later)
- **Delete** — VM and disks are deleted on eviction

Use Spot for batch jobs, rendering, CI/CD, and stateless web workers. Never for databases or any workload that cannot tolerate interruption.

### Azure Hybrid Benefit

If your organisation owns on-premises **Windows Server** or **SQL Server** licences with Software Assurance, you can apply them to Azure VMs — avoiding the OS/SQL licensing cost embedded in the VM price. This is unique to Azure and can save up to 49% on Windows VM costs.

## VM Availability Options

| Option | Protects against | SLA |
|---|---|---|
| **Single VM (Premium SSD)** | Hardware failures on the host | 99.9% |
| **Availability Set** | Rack-level hardware failure and planned maintenance within one datacentre | 99.95% |
| **Availability Zones** | Entire datacentre failure — VMs spread across 3 physical zones | 99.99% |
| **VM Scale Sets (Flexible)** | Distributes instances across zones automatically | 99.99% (with zones) |

### Key rules
- Availability Sets and Availability Zones **cannot be combined** on the same VM — choose one
- VMs in an Availability Set must be in the **same region**
- A VM cannot be moved into an Availability Set after creation — you must recreate it
- For new designs, **prefer Availability Zones** over Availability Sets

## VM States & Operations

| State | VM running? | Billed? | Disks preserved? |
|---|---|---|---|
| **Running** | Yes | Yes | Yes |
| **Stopped (OS-level)** | No | **Yes** | Yes |
| **Stopped (Deallocated)** | No | No (compute) | Yes |
| **Deleted** | No | No | Only if set to detach |

> **Critical:** Stopping a VM from within the OS does **not** stop billing. You must **Deallocate** the VM from the Azure portal, CLI, or SDK to stop compute charges. This is a common source of unexpected costs.

### Resizing

You can resize a VM to a different size at any time — the VM must be **deallocated** first. If the new size is not available in the same cluster as the existing VM, a new host is allocated (which may change the private IP if it was dynamic).

## Working with Azure VMs via Python

In [ ]:
# pip install azure-identity azure-mgmt-compute azure-mgmt-network azure-mgmt-resource

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.compute import ComputeManagementClient
from azure.mgmt.network import NetworkManagementClient

credential = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"

compute = ComputeManagementClient(credential, subscription_id)
network = NetworkManagementClient(credential, subscription_id)

In [ ]:
# List all VMs in the subscription with their size and power state
for vm in compute.virtual_machines.list_all():
    rg = vm.id.split('/')[4]
    iv = compute.virtual_machines.instance_view(rg, vm.name)
    statuses = [s.display_status for s in iv.statuses]
    power = next((s for s in statuses if 'VM' in s), 'unknown')
    print(f"{vm.name:<30} {vm.hardware_profile.vm_size:<25} {power}")

In [ ]:
# List available VM sizes in a region
sizes = compute.virtual_machine_sizes.list(location="eastus")
for s in sizes:
    if s.name.startswith("Standard_D"):   # filter to D-series
        print(f"{s.name:<30} vCPUs: {s.number_of_cores:<4} RAM: {s.memory_in_mb // 1024} GB")

In [ ]:
resource_group = "my-rg"
vm_name = "my-vm"

# Deallocate a VM (stops billing for compute)
poller = compute.virtual_machines.begin_deallocate(resource_group, vm_name)
poller.result()
print(f"{vm_name} deallocated")

# Start it again
poller = compute.virtual_machines.begin_start(resource_group, vm_name)
poller.result()
print(f"{vm_name} started")

In [ ]:
# List all managed disks and their types
for disk in compute.disks.list():
    print(f"{disk.name:<40} {disk.sku.name:<20} {disk.disk_size_gb} GB  "
          f"{'attached' if disk.managed_by else 'unattached'}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Images | Marketplace (OS + software), Azure Compute Gallery (custom versioned), Managed Image (one-off) |
| VM sizes | Series encodes the workload — B (burstable), D (general), E (memory), F (compute), N (GPU) |
| OS disk | Always a managed disk — persists across stop/start |
| Temporary disk | Local NVMe/SSD attached to host — lost on deallocation, resize, or host failure |
| Managed disk types | Standard HDD → Standard SSD → Premium SSD → Premium SSD v2 → Ultra Disk |
| NSG | Stateful firewall at subnet or NIC level — lower priority number = evaluated first |
| Custom data | Cloud-init or shell script — runs once on first boot |
| Extensions | Post-deployment automation — Custom Script, Azure Monitor Agent, Entra ID login |
| Spot VMs | Up to 90% off — evicted with 30s notice; eviction policy: Deallocate or Delete |
| Azure Hybrid Benefit | Apply on-prem Windows/SQL Server licences to Azure VMs — up to 49% savings |
| Deallocation | Must Deallocate (not just OS-stop) to stop compute billing |
| Availability | Single VM 99.9%, Availability Set 99.95%, Availability Zones 99.99% |